In [1]:
import pyodbc

In [2]:
def create_connection():
    # Thông tin kết nối SQL Server
    server = 'ZOHATEA'  # Tên server từ SSMS
    database = 'DW_Hotel'  # Đã sửa, bỏ dấu ;
    username = 'sa'
    password = 'ptit'

    # Kết nối SQL Server
    try:
        conn = pyodbc.connect(
            f"DRIVER={{ODBC Driver 17 for SQL Server}};"  # Sử dụng driver có sẵn
            f"SERVER={server};"
            f"DATABASE={database};"
            f"UID={username};"
            f"PWD={password}"
        )
        cursor = conn.cursor()
        print("Kết nối thành công!")
    except pyodbc.Error as ex:
        print(f"Lỗi kết nối: {ex}")
    return conn, cursor

# 1. Tạo dữ liệu cho Dim_Time (01/2024 → 12/2025)

In [3]:

import datetime
from datetime import timedelta
import pandas as pd

In [4]:
dim_time = []

for year in range(2024, 2026):
    for month in range(1, 13):
        time_key = int(f"{year}{month:02d}")
        quarter = (month - 1) // 3 + 1
        dim_time.append((time_key, year, quarter, month))

time_df = pd.DataFrame(dim_time, columns=['time_key', 'year', 'quarter', 'month'])
time_df.to_csv('dim_time.csv', index=False)

time_df       

,time_key,year,quarter,month
0,202401,2024,1,1
1,202402,2024,1,2
2,202403,2024,1,3
3,202404,2024,2,4
4,202405,2024,2,5
5,202406,2024,2,6
6,202407,2024,3,7
7,202408,2024,3,8
8,202409,2024,3,9
9,202410,2024,4,10


In [5]:
conn, cursor = create_connection()
# Chèn dữ liệu vào Dim_Time
try:
    cursor.executemany('''
    INSERT INTO Dim_Time (time_key, year, quarter, month)
    VALUES (?, ?, ?, ?)
    ''', dim_time)
    conn.commit()
    print(f"Dim_Time: {len(dim_time)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()

# Đóng kết nối
cursor.close()
conn.close()
print("Đã đóng kết nối.")   

Kết nối thành công!
Lỗi khi chèn dữ liệu: ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'Dim_Time'. (208) (SQLExecDirectW); [42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Statement(s) could not be prepared. (8180)")
Đã đóng kết nối.


# 2. Tạo dữ liệu cho Dim_Location

In [6]:
df = pd.read_csv('vietnam_provinces_regions.csv')
df.head(10)

,cityId,cityName,population,region
0,AGG,An Giang,1900000,Đồng bằng sông Cửu Long
1,BRV,Bà Rịa - Vũng Tàu,1150000,Đông Nam Bộ
2,BGG,Bắc Giang,1800000,Đồng bằng sông Hồng
3,BKN,Bắc Kạn,330000,Trung du và miền núi phía Bắc
4,BLU,Bạc Liêu,900000,Đồng bằng sông Cửu Long
5,BNH,Bắc Ninh,1400000,Đồng bằng sông Hồng
6,BTE,Bến Tre,1300000,Đồng bằng sông Cửu Long
7,BDH,Bình Định,1550000,Duyên hải Nam Trung Bộ
8,BDG,Bình Dương,2600000,Đông Nam Bộ
9,BPC,Bình Phước,1000000,Đông Nam Bộ


In [7]:
def generate_hotels_reduced(df, ratio=1/3):
    hotel_entries = []
    for _, row in df.iterrows():
        num_hotels = max(1, round((row["population"] / 100000) * ratio))
        for i in range(1, num_hotels + 1):
            hotel_id = f"{row['cityId']}{i:03d}"
            hotel_name = f"{row['cityName']} Hotel {i}"
            hotel_entries.append({
                "hotel_id": hotel_id,
                "cityId": row["cityId"],
                "hotelName": hotel_name,
                "cityName": row["cityName"],
                "region": row["region"]
            })
    return pd.DataFrame(hotel_entries)

In [8]:
df_dim_location = generate_hotels_reduced(df)
df_dim_location.to_csv('dim_location.csv', index=False)
df_dim_location

,hotel_id,cityId,hotelName,cityName,region
0,AGG001,AGG,An Giang Hotel 1,An Giang,Đồng bằng sông Cửu Long
1,AGG002,AGG,An Giang Hotel 2,An Giang,Đồng bằng sông Cửu Long
2,AGG003,AGG,An Giang Hotel 3,An Giang,Đồng bằng sông Cửu Long
3,AGG004,AGG,An Giang Hotel 4,An Giang,Đồng bằng sông Cửu Long
4,AGG005,AGG,An Giang Hotel 5,An Giang,Đồng bằng sông Cửu Long
...,...,...,...,...,...
272,VPC003,VPC,Vĩnh Phúc Hotel 3,Vĩnh Phúc,Đồng bằng sông Hồng
273,VPC004,VPC,Vĩnh Phúc Hotel 4,Vĩnh Phúc,Đồng bằng sông Hồng
274,YBI001,YBI,Yên Bái Hotel 1,Yên Bái,Trung du và miền núi phía Bắc
275,YBI002,YBI,Yên Bái Hotel 2,Yên Bái,Trung du và miền núi phía Bắc


In [9]:
# lưu vào dim_location
conn, cursor = create_connection()
# Chèn dữ liệu vào Dim_Location
try:
    cursor.executemany('''
    INSERT INTO Dim_Location (hotel_id, cityId, hotelName, cityName, region)
    VALUES (?, ?, ?, ?, ?)
    ''', df_dim_location.values.tolist())
    conn.commit()
    print(f"Dim_Location: {len(df_dim_location)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối  

Kết nối thành công!
Lỗi khi chèn dữ liệu: ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'Dim_Location'. (208) (SQLExecDirectW); [42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Statement(s) could not be prepared. (8180)")


# 3. Tạo dữ liệu cho Dim_Customer

In [10]:
from faker import Faker
import pandas as pd
import random
import numpy as np

In [11]:
# Tạo nhiều faker theo quốc tịch
fakers = {
    'Vietnam': Faker('vi_VN'),
    'USA': Faker('en_US'),
    'France': Faker('fr_FR'),
    'Japan': Faker('ja_JP'),
    'Korea': Faker('ko_KR'),
    'China': Faker('zh_CN'),
    'UK': Faker('en_GB'),
    'Germany': Faker('de_DE'),
    'Australia': Faker('en_AU')
}

nationalities = list(fakers.keys())
customer_types = ['Individual', 'Corporate', 'Travel Agency', 'VIP']

# Xác suất tương ứng (Vietnam chiếm 70%, còn lại chia đều 30%)
nationality_weights = [0.70] + [0.30 / (len(nationalities) - 1)] * (len(nationalities) - 1)

In [12]:
# Hàm sinh khách hàng
def generate_customers(num_customers=5_000):
    customer_data = []
    chosen_nationalities = np.random.choice(nationalities, size=num_customers, p=nationality_weights)
    
    for i, nationality in enumerate(chosen_nationalities, 1):
        faker = fakers[nationality]
        customer_key = f"CUST{i:05d}"
        customer_name = faker.name()
        customer_type = random.choice(customer_types)
        customer_data.append({
            "customer_key": customer_key,
            "customerName": customer_name,
            "customerType": customer_type,
            "nationality": nationality
        })
    
    return pd.DataFrame(customer_data)

In [13]:
df_dim_customers = generate_customers(5_000)
df_dim_customers.to_csv('dim_customer.csv', index=False)
df_dim_customers

,customer_key,customerName,customerType,nationality
0,CUST00001,Vi Mai,Individual,Vietnam
1,CUST00002,김성수,Travel Agency,Korea
2,CUST00003,Kevin Brown,Corporate,Australia
3,CUST00004,Bảo Vũ,Corporate,Vietnam
4,CUST00005,Bà Lâm Bùi,Corporate,Vietnam
...,...,...,...,...
4995,CUST04996,An Hoàng,Corporate,Vietnam
4996,CUST04997,佐藤 加奈,Corporate,Japan
4997,CUST04998,Trung Phạm,Individual,Vietnam
4998,CUST04999,Ông Trung Vũ,VIP,Vietnam


In [14]:
# lưu vào dim_customer
conn, cursor = create_connection()
# Chèn dữ liệu vào Dim_Customer
try:
    cursor.executemany('''
    INSERT INTO Dim_Customer (customer_key, customerName, customerType, nationality)
    VALUES (?, ?, ?, ?)
    ''', df_dim_customers.values.tolist())
    conn.commit()
    print(f"Dim_Customer: {len(df_dim_customers)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối  

Kết nối thành công!
Lỗi khi chèn dữ liệu: ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'Dim_Customer'. (208) (SQLExecDirectW); [42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Statement(s) could not be prepared. (8180)")


# 5. Tạo dữ liệu cho Dim_Room

In [15]:
# Danh sách các loại phòng phổ biến
room_types = [
    "Standard Room",
    "Deluxe Room",
    "Superior Room",
    "Executive Room",
    "Junior Suite",
    "Suite",
    "Presidential Suite",
    "Family Room",
    "Connecting Room",
    "Twin Room",
    "Double Room",
    "Single Room",
    "Studio Room",
    "Accessible Room",
    "Penthouse Suite"
]

# Tạo DataFrame
df_dim_rooms = pd.DataFrame({
    "roomType": room_types
})

In [16]:

df_dim_rooms.to_csv('dim_room.csv', index=False)
df_dim_rooms

,roomType
0,Standard Room
1,Deluxe Room
2,Superior Room
3,Executive Room
4,Junior Suite
5,Suite
6,Presidential Suite
7,Family Room
8,Connecting Room
9,Twin Room


In [17]:
# Lưu vào dim_room
conn, cursor = create_connection()
# Chèn dữ liệu vào Dim_Room
try:
    cursor.executemany('''
    INSERT INTO Dim_Room (roomType)
    VALUES (?)
    ''', df_dim_rooms.values.tolist())
    conn.commit()
    print(f"Dim_Room: {len(df_dim_rooms)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối  
cursor.close()

Kết nối thành công!
Lỗi khi chèn dữ liệu: ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'Dim_Room'. (208) (SQLExecDirectW); [42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Statement(s) could not be prepared. (8180)")


# 5. Tạo dữ liệu cho Dim_Service

In [18]:

# Định nghĩa các loại dịch vụ và tên dịch vụ tương ứng
services = {
    "Room Service": [
        "In-room Dining", "Mini Bar", "Daily Cleaning"
    ],
    "Wellness": [
        "Spa Treatment", "Massage Therapy", "Sauna", "Yoga Class"
    ],
    "Recreation": [
        "Swimming Pool Access", "Fitness Center", "Tennis Court"
    ],
    "Food & Beverage": [
        "Breakfast Buffet", "Restaurant Dining", "Bar & Lounge"
    ],
    "Business": [
        "Conference Room", "Business Center", "Printing Services"
    ],
    "Transportation": [
        "Airport Shuttle", "Car Rental", "Bicycle Rental"
    ],
    "Concierge": [
        "Tour Booking", "Luggage Storage", "Currency Exchange"
    ],
    "Other": [
        "Pet Service", "Laundry Service", "Babysitting"
    ]
}

# Tạo dữ liệu
service_data = []
service_counter = 1

for service_type, service_names in services.items():
    for name in service_names:
        service_key = f"SVC{service_counter:03d}"
        service_data.append({
            "service_key": service_key,
            "serviceName": name,
            "serviceType": service_type
        })
        service_counter += 1

df_dim_services = pd.DataFrame(service_data)
df_dim_services.to_csv('dim_service.csv', index=False)
df_dim_services

,service_key,serviceName,serviceType
0,SVC001,In-room Dining,Room Service
1,SVC002,Mini Bar,Room Service
2,SVC003,Daily Cleaning,Room Service
3,SVC004,Spa Treatment,Wellness
4,SVC005,Massage Therapy,Wellness
5,SVC006,Sauna,Wellness
6,SVC007,Yoga Class,Wellness
7,SVC008,Swimming Pool Access,Recreation
8,SVC009,Fitness Center,Recreation
9,SVC010,Tennis Court,Recreation


In [19]:
# Lưu vào dim_service
conn, cursor = create_connection()
# Chèn dữ liệu vào Dim_Service
try:
    cursor.executemany('''
    INSERT INTO Dim_Service (service_key, serviceName, serviceType)
    VALUES (?, ?, ?)
    ''', df_dim_services.values.tolist())
    conn.commit()
    print(f"Dim_Service: {len(df_dim_services)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối
cursor.close()

Kết nối thành công!
Lỗi khi chèn dữ liệu: ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'Dim_Service'. (208) (SQLExecDirectW); [42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Statement(s) could not be prepared. (8180)")


# 6. Tạo dữ liệu cho Fact_Room

In [20]:
import pandas as pd
import random

# Tạo danh sách key từ index + 1
location_keys = list(range(1, len(df_dim_location) + 1))
room_keys = list(range(1, len(df_dim_rooms) + 1))
customer_keys = df_dim_customers["customer_key"].tolist()
time_keys = time_df['time_key'].tolist()

fact_data = []

for _ in range(50_000):  # số dòng tuỳ chỉnh
    time_key = random.choice(time_keys)
    location_key = random.choice(location_keys)
    room_key = random.choice(room_keys)
    customer_key = random.choice(customer_keys)

    booking_count = random.choices([1, 2, 3, 4, 5], weights=[0.2, 0.3, 0.25, 0.15, 0.1])[0]

    if booking_count == 0:
        revenue = 0.0
        rating = None
    else:
        unit_price = random.randint(1_000_0000, 10_000_000)
        revenue = round(booking_count * unit_price, 2)
        rating = round(random.uniform(2.5, 5.0), 2)

    fact_data.append({
        "time_key": time_key,
        "location_key": location_key,
        "room_key": room_key,
        "customer_key": customer_key,
        "roomRevenue": revenue,
        "roomBookingCount": booking_count,
        "avgRating": rating
    })

df_fact_room = pd.DataFrame(fact_data)

# Gộp theo bộ key để đảm bảo duy nhất
df_fact_room = df_fact_room.groupby(
    ["time_key", "location_key", "room_key", "customer_key"], as_index=False
).agg({
    "roomRevenue": "sum",
    "roomBookingCount": "sum",
    "avgRating": "mean"
})


df_fact_room = df_fact_room.sort_values(by='time_key', ascending=True)
df_fact_room.to_csv('fact_room.csv', index=False)
df_fact_room

,time_key,location_key,room_key,customer_key,roomRevenue,roomBookingCount,avgRating
19,202401,3,15,CUST04507,20000000,2,2.70
18,202401,3,13,CUST00826,10000000,1,4.87
17,202401,3,12,CUST00643,10000000,1,3.90
16,202401,3,10,CUST04383,30000000,3,3.24
15,202401,3,10,CUST02698,20000000,2,4.89
...,...,...,...,...,...,...,...
49974,202512,275,10,CUST01162,50000000,5,3.92
49975,202512,275,10,CUST01374,30000000,3,2.55
49976,202512,275,13,CUST02866,30000000,3,4.83
49977,202512,275,15,CUST02073,30000000,3,4.00


In [21]:
# lưu vào fact_room
conn, cursor = create_connection()
# Chèn dữ liệu vào Fact_Room
try:
    cursor.executemany('''
    INSERT INTO Fact_Room (time_key, location_key, room_key, customer_key, roomRevenue, roomBookingCount, avgRating)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', df_fact_room.values.tolist())
    conn.commit()
    print(f"Fact_Room: {len(df_fact_room)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback() 
# Đóng kết nối
cursor.close()

Kết nối thành công!
Lỗi khi chèn dữ liệu: ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'Fact_Room'. (208) (SQLExecDirectW); [42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Statement(s) could not be prepared. (8180)")


# 7. tạo dữ liệu Face_service

In [22]:
import pandas as pd
import random

# Tạo các khóa một lần duy nhất
location_keys = list(range(1, len(df_dim_location) + 1))
service_df = df_dim_services.copy()
customer_keys = df_dim_customers["customer_key"].tolist()
time_keys = time_df["time_key"].tolist()

# Map giá theo loại dịch vụ
service_price_map = {
    "Spa": (300_000, 1_500_000),
    "Laundry": (50_000, 300_000),
    "Transport": (200_000, 1_000_000),
    "Food & Beverage": (100_000, 2_000_000),
    "Gym": (0, 200_000),
    "Tour": (500_000, 3_000_000),
    "Other": (100_000, 500_000)
}

# Lấy danh sách services thành list để tránh sample nhiều lần
services_list = service_df[["service_key", "serviceType"]].values.tolist()

fact_service_data = []

for _ in range(50_000):
    time_key = random.choice(time_keys)
    location_key = random.choice(location_keys)
    service_key, service_type = random.choice(services_list)
    customer_key = random.choice(customer_keys)

    service_count = random.choices([1, 2, 3, 4], weights=[0.3, 0.4, 0.2, 0.1])[0]

    if service_count == 0:
        revenue = 0.0
    else:
        min_price, max_price = service_price_map.get(service_type, (100_000, 500_000))
        unit_price = random.randint(min_price, max_price)
        revenue = round(service_count * unit_price, 2)

    fact_service_data.append({
        "time_key": time_key,
        "location_key": location_key,
        "service_key": service_key,
        "customer_key": customer_key,
        "serviceRevenue": revenue,
        "serviceCount": service_count
    })


df_fact_service = pd.DataFrame(fact_service_data)

# Gộp các dòng trùng bộ key
df_fact_service = df_fact_service.groupby(
    ["time_key", "location_key", "service_key", "customer_key"], as_index=False
).agg({
    "serviceRevenue": "sum",
    "serviceCount": "sum"
})

df_fact_service = df_fact_service.sort_values(by='time_key', ascending=True)
df_fact_service.to_csv('fact_service.csv', index=False)
df_fact_service

,time_key,location_key,service_key,customer_key,serviceRevenue,serviceCount
5,202401,1,SVC013,CUST04666,2568058,2
4,202401,1,SVC012,CUST00176,289146,1
3,202401,1,SVC011,CUST03309,3026028,2
2,202401,1,SVC009,CUST02199,898946,2
1,202401,1,SVC008,CUST03692,398738,2
...,...,...,...,...,...,...
49970,202512,273,SVC024,CUST03815,834092,2
49969,202512,273,SVC023,CUST00369,808754,2
49999,202512,277,SVC025,CUST03800,431599,1
49968,202512,273,SVC020,CUST04732,401543,1


In [23]:
# Lưu vào fact_service
conn, cursor = create_connection()
# Chèn dữ liệu vào Fact_Service
try:
    cursor.executemany('''
    INSERT INTO Fact_Service (time_key, location_key, service_key, customer_key, serviceRevenue, serviceCount)
    VALUES (?, ?, ?, ?, ?, ?)
    ''', df_fact_service.values.tolist())
    conn.commit()
    print(f"Fact_Service: {len(df_fact_service)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối
cursor.close()

Kết nối thành công!
Lỗi khi chèn dữ liệu: ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'Fact_Service'. (208) (SQLExecDirectW); [42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Statement(s) could not be prepared. (8180)")


# 8. tạo dữ liệu fact_revenue

In [24]:
# Merge revenue từ room và service
room_rev = df_fact_room.groupby(
    ["time_key", "location_key", "customer_key"], as_index=False
)["roomRevenue"].sum()

service_rev = df_fact_service.groupby(
    ["time_key", "location_key", "customer_key"], as_index=False
)["serviceRevenue"].sum()

# Kết hợp 2 bảng (outer join để không mất dữ liệu)
df_fact_revenue = pd.merge(room_rev, service_rev, 
                           on=["time_key", "location_key", "customer_key"], 
                           how="outer")

# Fill NA với 0 (nếu chỉ có ở một bảng)
df_fact_revenue["roomRevenue"] = df_fact_revenue["roomRevenue"].fillna(0)
df_fact_revenue["serviceRevenue"] = df_fact_revenue["serviceRevenue"].fillna(0)

# Tính tổng doanh thu
df_fact_revenue["totalRevenue"] = df_fact_revenue["roomRevenue"] + df_fact_revenue["serviceRevenue"]

# Giữ cột cần thiết
df_fact_revenue = df_fact_revenue[["time_key", "location_key", "customer_key", "totalRevenue"]]
df_fact_revenue = df_fact_revenue.sort_values(by="time_key")

# Lưu file
df_fact_revenue.to_csv("fact_revenue.csv", index=False)
df_fact_revenue.head()


,time_key,location_key,customer_key,totalRevenue
25,202401,3,CUST00307,442238.0
24,202401,3,CUST00278,20000000.0
23,202401,3,CUST00169,10000000.0
22,202401,2,CUST04613,30000000.0
21,202401,2,CUST04212,922082.0


In [25]:
# Lưu vào fact_revenue
conn, cursor = create_connection()
# Chèn dữ liệu vào Fact_Revenue
try:
    cursor.executemany('''
    INSERT INTO Fact_Revenue (time_key, location_key, customer_key, totalRevenue)
    VALUES (?, ?, ?, ?)
    ''', df_fact_revenue.values.tolist())
    conn.commit()
    print(f"Fact_Revenue: {len(df_fact_revenue)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối
cursor.close()

Kết nối thành công!
Lỗi khi chèn dữ liệu: ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'Fact_Revenue'. (208) (SQLExecDirectW); [42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Statement(s) could not be prepared. (8180)")
